# First simulation with OpenMM

## 1. Force field definition and energy minimization

The first simulation below is a minimal example of a molecular system: just a single water molecule. Initially, only the energy of this system is minimized.
This example consists of a few steps:

### Import Python packages

OpenMM must be imported before one can use it.
The other packages are used to define an array with initial positions and to analyze results (NumPy),
to load and plot simulation output (Pandas and Matplotlib)
and to visualize structures (MDTraj and NGLView).


In [ ]:
%matplotlib widget
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np
import openmm as mm
import pandas
from openmm import app, unit

### Units

OpenMM uses internally consistent units based on the SI system, as documented [here](https://docs.openmm.org/latest/userguide/theory/01_introduction.html#units).
One may easily convert values with different units, as shown in the examples below.

In [ ]:
# Assign a value of 10 picoseconds
time = 10 * unit.picosecond  # 10 * unit.picoseconds will also work.
print("Unit of variable time:", time.unit)
print("time:", time)
print("time [s]:", time / unit.second)
print("time [s]:", time / unit.seconds)
print("time [fs]:", time.value_in_unit(unit.femtosecond))
print("time:", time.in_units_of(unit.femtosecond))

# OpenMM also knows a few important constants.
print("Boltzmann's constant:", unit.BOLTZMANN_CONSTANT_kB)
print("Avogadro's constant:", unit.AVOGADRO_CONSTANT_NA)

The API documentation for quantities with units can be found here:

https://docs.openmm.org/latest/api-python/generated/openmm.unit.quantity.Quantity.html

A complete overview of units and constants can be found in the Python wrapper source code for OpenMM:

* Units: https://github.com/openmm/openmm/blob/master/wrappers/python/openmm/unit/unit_definitions.py
* Constants: https://github.com/openmm/openmm/blob/master/wrappers/python/openmm/unit/constants.py

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> - Try to change (femto)second(s) to (femto)meter(s) in the above example. Why does this work or fail in each case?
>
> - Use the Boltzmann constant and the Avogadro constant to express the universal gas constant $R$ in $\textrm{kJ}\,\textrm{K}^{-1}\,\textrm{mol}^{-1}$.
>
> - Print $RT$ in $\textrm{kJ}\,\textrm{mol}^{-1}$, where $T=293.15\,\textrm{K}$. This is the molar equivalent of $k_B T$: the average amount of (kinetic and potential) thermal energy per degree of freedom in a system, at room temperature.

### Define the topology

In normal OpenMM usage, the topology is defined by a PDB file.
In this example, it is defined from scratch by creating objects in Python code, to show explicitly how the topology is built up.
Similar data structures are created when loading topologies from files.

Note that a topology in biomolecular simulations consists of a group of chains, where a chain consists of one or more residues and every residue contains a set of atoms, which are connected by bonds.
This terminology was originally defined for describing proteins in the PDB file format, see https://www.wwpdb.org/documentation/file-format.
In biomolecular simulations, the same terminology is nowadays also used for other things than proteins.
For example, here we have a chain with a single residue, a water molecule, containing three atoms and two bonds.

In [ ]:
topology = app.Topology()
chain = topology.addChain()
residue = topology.addResidue("water", chain)
element_O = app.Element.getByAtomicNumber(8)
element_H = app.Element.getByAtomicNumber(1)
atom0 = topology.addAtom("O", element_O, residue)
atom1 = topology.addAtom("H", element_H, residue)
atom2 = topology.addAtom("H", element_H, residue)
topology.addBond(atom0, atom1)
topology.addBond(atom0, atom2)

### Set up the simulation

This part consists of a few steps:

1. Select the TIP3P-FB **force field definition** for water, see https://doi.org/10.1021/jz500737m. The file `amber14/tip3pfb.xml` is included in the OpenMM installation. Its contents can also be viewed online, see https://github.com/openmm/openmm/blob/master/wrappers/python/openmm/app/data/amber14/tip3pfb.xml

1. Define a **system**, which is essentially an object implementing the selected force field for our topology. This object can compute energies and forces.

1. Define an **integrator**, which implements a molecular dynamics algorithm, in this case the Verlet algorithm sampling the NVE ensemble. Even though we will initially only carry out a geometry optimization, OpenMM requires us to define an integrator.

1. Define a **simulation**, which has the following responsibilities:

   - keep track of the current state (atomic positions etc.),

   - decide which implementation to use (CPU versus GPU)

   - drive the integrator and

   - write output files (reporters).

No reporters are attached to the simulation yet: these will be added in section 4 to write output during the molecular dynamics run. The initial positions are provided as a NumPy array, multiplied by a unit. Each row in the array contains the X, Y and Z coordinates of one atomic nucleus, in the same order as the atoms in the topology. Normally, also these initial positions are loaded from a PDB file.

In [ ]:
# 1. Select the TIP3P-FB force field.
forcefield = app.ForceField("amber14/tip3pfb.xml")

# 2. Create an object to compute energies and forces for our topology.
system = forcefield.createSystem(topology)

# 3. Define an integrator, mandatory.
integrator = mm.VerletIntegrator(1 * unit.femtoseconds)

# 4. A simulation object in OpenMM combines topology, system and integrator.
simulation = app.Simulation(topology, system, integrator)
simulation.context.setPositions(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [0.0, 0.0, 1.0],
            [0.0, 1.0, 0.0],
        ]
    )
    * unit.angstroms
)

### Minimize the energy

Also this part is split up in a few steps:

1. Before minimizing, compute and print the **energy** of the **initial** state.

1. **Minimize** the energy, using the L-BFGS algorithm.
   This is an iterative procedure: in every iteration the atomic positions are adjusted to lower the energy.
   To follow the progress, we attach a **minimization reporter**, which OpenMM calls after every iteration.
   Here, it writes the energy as function of the iteration number to a CSV file.
   See the [OpenMM user guide](https://docs.openmm.org/latest/userguide/application/02_running_sims.html#energy-minimization) for more details.

1. Compute and print the **energy** of the **final** state.

The reporter is defined first.
Unlike the reporters used for molecular dynamics (see section 4 below), OpenMM has no ready-made reporter for minimizations.
We write a small one ourselves, by defining a Python class that inherits from `MinimizationReporter`.
The only method OpenMM requires is `report`, which receives the iteration number, the current positions `x` and gradient `grad` (unused here) and a dictionary `args` with some extra information.
The potential energy is stored in `args["system energy"]`, in $\textrm{kJ}\,\textrm{mol}^{-1}$.
If `report` returns `True`, the minimization is stopped early.

In [ ]:
class CSVMinimizationReporter(mm.MinimizationReporter):
    """Write the energy after every minimization iteration to a CSV file."""

    def __init__(self, filename):
        super().__init__()
        self.file = open(filename, "w")
        self.file.write("Iteration,Energy (kJ/mole)\n")

    def report(self, iteration, x, grad, args):
        self.file.write(f"{iteration},{args['system energy']}\n")
        # Returning True would stop the minimization.
        return False

    def close(self):
        self.file.close()

In [ ]:
# 1. Print the energy before minimizing.
state0 = simulation.context.getState(getEnergy=True)
print(state0.getPotentialEnergy())

# 2. Minimize the energy and write the energy of each iteration to a CSV file.
reporter = CSVMinimizationReporter("minimization.csv")
simulation.minimizeEnergy(reporter=reporter)
reporter.close()

# 3. Print the energy afterwards.
state1 = simulation.context.getState(getEnergy=True)
print(state1.getPotentialEnergy())

The CSV file can be loaded with Pandas and plotted, just like the output of the molecular dynamics simulation in section 4 below.
Note that the reporter is first called after the first iteration, so the initial energy printed above is not included in the file.

In [ ]:
df_min = pandas.read_csv("minimization.csv")
df_min

In [ ]:
# Plot the energy as function of the minimizer iteration.
# The semicolon at the end suppresses the textual output of the last line.
plt.close("minimization")
fig, ax = plt.subplots(num="minimization")
df_min.plot(kind="line", x="Iteration", y="Energy (kJ/mole)", marker="o", ax=ax)
ax.set_xticks(df_min["Iteration"]);

For this tiny system, the minimizer converges after just a few iterations.
The energy drops by several orders of magnitude, from a rather high value for the artificial initial geometry (an H-O-H angle of $90^\circ$) to practically zero.
The energy is zero at the minimum, because the water molecule has no other interactions than the bond and angle terms.
For larger molecules, many more iterations are needed, as we will see in the tutorial on the villin headpiece.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> - The minimizer stops when the root-mean-square of all force components drops below a tolerance, which is $10\,\textrm{kJ}\,\textrm{mol}^{-1}\,\textrm{nm}^{-1}$ by default.
>   Pass a smaller `tolerance` to `simulation.minimizeEnergy()`, e.g. `tolerance=0.01 * unit.kilojoules_per_mole / unit.nanometer`, and rerun the cells above. How does the number of iterations change?
>   The API documentation can be found here: https://docs.openmm.org/latest/api-python/generated/openmm.app.simulation.Simulation.html
>
> - Add `ax.set_yscale("log")` to the plotting cell. Why is a logarithmic scale useful here?

## 2. Inspection of OpenMM data structures

Before performing other simulations, we take a closer look at the objects created in the previous example to understand how OpenMM works. To access the internals, we use the so-called Python Application Programming Interface (API) of OpenMM, which is documented here: https://docs.openmm.org/latest/api-python/index.html.

The API documentation contains tables with class names, like `Topology`, and their corresponding purpose. Clicking on the class name brings you to a page with more details.

### Topology

The `Topology` class is documented here: https://docs.openmm.org/latest/api-python/generated/openmm.app.topology.Topology.html#openmm.app.topology.Topology

In [ ]:
# The print lines below show details of the topology.
print(topology)
print("Number of atoms:", topology.getNumAtoms())
print("Periodic boundaries:", topology.getUnitCellDimensions())
for iatom, atom in enumerate(topology.atoms()):
    # The following line would just print a dictionary
    # with all attributes, useful to explore undocumented
    # attributes.
    # print(vars(atom))
    print(f"Name and mass of atom {iatom}: {atom.name}, {atom.element.mass}")

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Extend the code cell above to loop over the bonds and to print the corresponding pair of atoms.

### System

The relevant API documentation can be found here:

* `System` class: https://docs.openmm.org/latest/api-python/generated/openmm.openmm.System.html
* `HarmonicBondForce` class: https://docs.openmm.org/latest/api-python/generated/openmm.openmm.HarmonicBondForce.html

In [ ]:
# Loop over all contributions to the force evaluation.
# Note that not all terms are real force-field contributions:
# the CMMotionRemover zeroes the center-of-mass momentum at every step.
for force in system.getForces():
    print(force)
print()
# The force field contains two harmonic bond terms,
# with the same parameters.
force_bond = system.getForce(0)
for ibond in range(force_bond.getNumBonds()):
    print(force_bond.getBondParameters(ibond))
    print()

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Add code to show the angle force terms of the system and the corresponding parameters. Do these match the definition of the force field in the `tip3pfb.xml` file?

### System state

A `Simulation` object has a `context` attribute, through which many details of the simulation can be accessed. Here we use it to retrieve the current state of our molecular model. The `Context` class is documented here: https://docs.openmm.org/latest/api-python/generated/openmm.openmm.Context.html

Below, the `getState` method retrieves a `State` object with results for the final geometry of the water molecule. The `State` class is documented here: https://docs.openmm.org/latest/api-python/generated/openmm.openmm.State.html

In [ ]:
# Simply extract and print information
state = simulation.context.getState(getPositions=True, getForces=True, getEnergy=True)
print("Potential energy: ", state.getPotentialEnergy())
print()
print("Atomic positions")
print(state.getPositions(asNumpy=True))
print()
print("Forces")
print(state.getForces(asNumpy=True))
print()

In [ ]:
# One can use NumPy to get a few geometry properties:

# The length of the first O-H bond, in nanometers.
pos = state.getPositions(asNumpy=True)
print(np.linalg.norm(pos[1] - pos[0]))

# The H-O-H angle, in degrees
d01 = pos[1] - pos[0]
d02 = pos[2] - pos[0]
d01 /= np.linalg.norm(d01)
d02 /= np.linalg.norm(d02)
cosine = np.dot(d01, d02)
print(np.arccos(cosine) * 180 / np.pi)

## 3. PDB output and visualization

The following is a minimal example writing out the current positions to a PDB file, after which NGLView is used to visualize the structure. See https://nglviewer.org/nglview/latest/.

In [ ]:
# NGLView is a structure and trajectory visualization tool
# that integrates well with Jupyter Notebooks.

# Write out a PDB file
with open("water.pdb", "w") as outfile:
    app.PDBFile.writeFile(topology, pos, outfile)
# Visualize
nglview.show_mdtraj(mdtraj.load("water.pdb"))

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Check the contents of the file `water.pdb`. It should contain the Cartesian coordinates of the optimized geometry.

## 4. A short molecular dynamics simulation

In the code cells above, a Verlet integrator was defined, which we will use in this section.

Before performing the MD, some more initialization steps are needed:

- Velocities are assigned random values sampled from the Maxwell-Boltzmann distribution.

- Three reporters are attached to the simulation to store the output in files and to track the progress on screen. The reporter API is documented here: https://docs.openmm.org/latest/api-python/app.html#reporting-output

In [ ]:
# Initialize velocities with random values at 300K.
# The random seed is fixed (to 1). This way, rerunning
# this cell will generate the same random velocities.
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin, 1)

# Remove existing reporters,
# in case this cell is executed more than once.
simulation.reporters = []

# Write a frame to the PDB trajectory every step.
simulation.reporters.append(app.PDBReporter("traj.pdb", 1))

# Write scalar properties to a CSV file every step.
simulation.reporters.append(
    app.StateDataReporter(
        "scalars.csv",
        1,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)

# Write scalar properties to screen every 100 steps.
simulation.reporters.append(
    app.StateDataReporter(stdout, 100, step=True, totalEnergy=True, temperature=True)
)

# Actually run the molecular dynamics simulation.
simulation.step(1000)

We will load the trajectory again with MDTraj and pass on this information to nglview to visualize the motion of the water molecule. See https://www.mdtraj.org/.

In [ ]:
nglview.show_mdtraj(mdtraj.load("traj.pdb"))

The following example shows how to load the data from the CSV file and plot it, using Pandas. Under the hood, Pandas uses Matplotlib to make the figure. In the example below, two curves are put on the same plot by plotting them in the same Matplotlib axes.

See https://pandas.pydata.org/ and https://matplotlib.org/.

Because OpenMM writes slightly non-standard CSV output, the header of the first column contains a few spurious characters.
These are removed by renaming the column after loading the file.

In [ ]:
# Load the CSV file into a Pandas DataFrame.
df = pandas.read_csv("scalars.csv")
df = df.rename(columns={'#"Time (ps)"': "Time (ps)"})
# To show the contents of a Pandas Dataframe, just put
# the variable on the last line.
df

In [ ]:
# Plot the potential and total energy as function of time.
plt.close("energy")
fig, ax = plt.subplots(num="energy")
df.plot(kind="line", x="Time (ps)", y="Potential Energy (kJ/mole)", ax=ax)
df.plot(kind="line", x="Time (ps)", y="Total Energy (kJ/mole)", ax=ax);

In [ ]:
plt.close("temperature")
fig, ax = plt.subplots(num="temperature")
df.plot(kind="line", x="Time (ps)", y="Temperature (K)", ax=ax);

The oscillations in the potential energy are normal. Due to the molecular vibrations, there is a continuous exchange between kinetic and potential energy. In principle, the total energy should be conserved exactly. In simulations, it is only approximately conserved because the Verlet integrator uses a finite time step. For practical applications, it is sufficient to observe no systematic drift in the conserved quantity.

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Experiment with this notebook to get some experience with molecular dynamics simulations:
>
> - The velocities were initialized at a temperature of $300\,\textrm{K}$, but the average temperature during the simulation is significantly lower.
>   Can you explain why?
>
> - The timestep is set to $1\,\textrm{fs}$ when creating the `VerletIntegrator` object. Increase the time step until the total energy is no longer conserved. What happens?
>
> - Replace the `VerletIntegrator` by a `LangevinMiddleIntegrator` at a temperature of $300\,\textrm{K}$, with a friction coefficient of $10\,\textrm{ps}^{-1}$ and a step size of $1\,\textrm{fs}$. The API documentation can be found here: [LangevinMiddleIntegrator](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.LangevinMiddleIntegrator.html).
>
> - Make modifications to the topology to understand which parts are essential to generate the system object with the correct parameters. Try making changes to: (i) the atom names, (ii) the element numbers, (iii) the bonds, (iv) the name of the residue and (v) the order of the atoms.